# ProfitPilot Kaggle T4 x2 5m Multi-Asset PPO Training

Built for **Kaggle Notebooks with the GPU T4 x2 accelerator**.

This notebook trains one Stable-Baselines3/RecurrentPPO model on `cuda:0` and uses multiple
parallel rollout environments to improve wall-clock time versus the old single-env T4 run.
It also verifies that both Kaggle T4 GPUs are visible. The second T4 is not used for one-model
SB3 updates; using both GPUs directly requires running a second training job or a distributed
training refactor.

It downloads BTC/USDT and ETH/USDT 5-minute candles from `2024-01-01` through `2026-04-28`,
trains the RL trading model on the `2024-01-01` to `2026-03-31` portion,
and tests out of sample on `2026-04-01` through `2026-04-28`.

### Kaggle setup
1. In Kaggle Notebook settings, set the **Accelerator** to **GPU T4 x2** and enable **Internet**.
2. Run the clone/setup cells, or add the ProfitPilot project as a Kaggle Dataset.
3. Optional: set Kaggle Secret `PROFIT_PILOT_ROOT` to a mounted project path.
4. Optional speed knobs: `PROFIT_PILOT_TRAIN_N_ENVS`, `PROFIT_PILOT_N_STEPS`, and `PROFIT_PILOT_BATCH_SIZE`.

All outputs (model, reports, data) are written to `/kaggle/working/` and can be downloaded or saved as a dataset.


In [1]:
# Install project dependencies without overwriting Kaggle's pre-installed CUDA PyTorch build.
%pip install -q ccxt ta stable-baselines3 sb3-contrib gymnasium pyarrow seaborn python-dotenv PyYAML


Note: you may need to restart the kernel to use updated packages.


In [2]:
!rm -rf /kaggle/working/ProfitPilot
!git clone --depth 1 --branch kaggle-run https://github.com/DivyeshVagh7/Profit-Pilot.git /kaggle/working/ProfitPilot
%cd /kaggle/working/ProfitPilot


Cloning into '/kaggle/working/ProfitPilot'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 175 (delta 38), reused 148 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 195.98 MiB | 27.92 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Updating files: 100% (122/122), done.
/kaggle/working/ProfitPilot


In [3]:
%cd /kaggle/working/ProfitPilot
!git pull


/kaggle/working/ProfitPilot
Already up to date.


In [4]:
%cd /kaggle/working/ProfitPilot


/kaggle/working/ProfitPilot


In [5]:
import sys, os
PROJECT_ROOT = "/kaggle/working/ProfitPilot"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, f"{PROJECT_ROOT}/src")

print("Project root:", os.getcwd())


Project root: /kaggle/working/ProfitPilot


In [6]:
from pathlib import Path

print(Path("src/profit_pilot").exists())
print(list(Path("src/profit_pilot").iterdir())[:10])


True
[PosixPath('src/profit_pilot/features'), PosixPath('src/profit_pilot/train'), PosixPath('src/profit_pilot/env'), PosixPath('src/profit_pilot/config.py'), PosixPath('src/profit_pilot/data'), PosixPath('src/profit_pilot/utils'), PosixPath('src/profit_pilot/__init__.py')]


In [7]:
from pathlib import Path
import json
import math
import os
import random
import sys
import warnings

# ---------------------------------------------------------------------------
# Project root detection - robust Kaggle-first search
# ---------------------------------------------------------------------------

def _find_profit_pilot_root() -> Path | None:
    """Search for the ProfitPilot project root that contains src/profit_pilot."""

    def is_valid_root(p: Path) -> bool:
        try:
            return (p / 'src' / 'profit_pilot').exists()
        except OSError:
            return False

    env_root = os.environ.get('PROFIT_PILOT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if is_valid_root(candidate):
            return candidate

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for top in sorted(kaggle_input.iterdir()):
            if is_valid_root(top):
                return top.resolve()
            try:
                for sub in sorted(top.iterdir()):
                    if sub.is_dir() and is_valid_root(sub):
                        return sub.resolve()
            except PermissionError:
                pass

    for candidate in [Path.cwd(), *Path.cwd().parents]:
        try:
            if is_valid_root(candidate.resolve()):
                return candidate.resolve()
        except OSError:
            pass

    return None


PROJECT_ROOT = _find_profit_pilot_root()

if PROJECT_ROOT is None:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        available = [str(p) for p in sorted(kaggle_input.iterdir())]
        print('Available Kaggle input paths:')
        for p in available:
            print(f'  {p}')
    raise FileNotFoundError(
        'Could not find ProfitPilot (src/profit_pilot not found anywhere under /kaggle/input/). '
        'Make sure you uploaded the project as a Kaggle Dataset, cloned it into /kaggle/working, '
        'or set PROFIT_PILOT_ROOT to the correct path.'
    )

WORKING_ROOT = Path(os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working'))
if not WORKING_ROOT.exists():
    WORKING_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root : {PROJECT_ROOT}')
print(f'Working root : {WORKING_ROOT}')

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from profit_pilot.utils.torch_device import select_torch_device

warnings.filterwarnings('ignore', category=FutureWarning)

EXPECTED_T4_GPU_COUNT = int(os.environ.get('PROFIT_PILOT_EXPECTED_T4_GPU_COUNT', '2'))
CUDA_TRAIN_DEVICE = 'cuda:0'
CUDA_TRAIN_DEVICE = os.environ.get('PROFIT_PILOT_CUDA_TRAIN_DEVICE', CUDA_TRAIN_DEVICE)

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
AVAILABLE_GPU_NAMES = []
if GPU_COUNT:
    print(f'Visible CUDA GPUs: {GPU_COUNT}')
    for gpu_index in range(GPU_COUNT):
        gpu_name = torch.cuda.get_device_name(gpu_index)
        AVAILABLE_GPU_NAMES.append(gpu_name)
        capability = torch.cuda.get_device_capability(gpu_index)
        print(f'  cuda:{gpu_index}: {gpu_name} (sm_{capability[0]}{capability[1]})')
else:
    print('Visible CUDA GPUs: 0')

if GPU_COUNT < EXPECTED_T4_GPU_COUNT:
    print(
        f'Warning: expected {EXPECTED_T4_GPU_COUNT} Kaggle T4 GPUs but only {GPU_COUNT} CUDA device(s) are visible. '
        'Check Kaggle Notebook settings and select GPU T4 x2.'
    )
elif not all('T4' in name.upper() for name in AVAILABLE_GPU_NAMES[:EXPECTED_T4_GPU_COUNT]):
    print(
        'Warning: two CUDA devices are visible, but they do not all look like Tesla T4 GPUs: '
        f'{AVAILABLE_GPU_NAMES}'
    )
else:
    print('Kaggle T4 x2 check: both expected T4 GPUs are visible.')

DEVICE_SELECTION = select_torch_device(
    torch,
    preferred=os.environ.get('PROFIT_PILOT_TORCH_DEVICE', 'auto'),
)
if DEVICE_SELECTION.device == 'cuda':
    torch.cuda.set_device(torch.device(CUDA_TRAIN_DEVICE).index or 0)
    DEVICE = CUDA_TRAIN_DEVICE
else:
    DEVICE = 'cpu'

print(f'PyTorch train device: {DEVICE}')
if DEVICE_SELECTION.gpu_name:
    print(f'Primary GPU: {DEVICE_SELECTION.gpu_name}')
if DEVICE_SELECTION.cuda_version:
    print(f'CUDA version: {DEVICE_SELECTION.cuda_version}')
if DEVICE_SELECTION.capability:
    print(f'Primary GPU capability: {DEVICE_SELECTION.capability}')
if DEVICE_SELECTION.arch_list:
    print(f'Torch CUDA arch list: {", ".join(DEVICE_SELECTION.arch_list)}')
print(f'Device decision: {DEVICE_SELECTION.reason}')
if DEVICE == CUDA_TRAIN_DEVICE:
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')
else:
    print('CUDA is unavailable or unusable; training will run on CPU. Use Kaggle GPU T4 x2 for this notebook.')


Project root : /kaggle/working/ProfitPilot
Working root : /kaggle/working
Visible CUDA GPUs: 2
  cuda:0: Tesla T4 (sm_75)
  cuda:1: Tesla T4 (sm_75)
Kaggle T4 x2 check: both expected T4 GPUs are visible.
PyTorch train device: cuda:0
Primary GPU: Tesla T4
CUDA version: 12.8
Primary GPU capability: sm_75
Torch CUDA arch list: sm_70, sm_75, sm_80, sm_86, sm_90, sm_100, sm_120
Device decision: CUDA smoke test passed.


In [8]:
import importlib
import inspect
import time

import ccxt
import matplotlib.pyplot as plt
import seaborn as sns
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecMonitor

import profit_pilot.data.download_ohlcv as download_ohlcv_module
import profit_pilot.env.multi_crypto_env as multi_crypto_env_module
import profit_pilot.features.build_features as build_features_module
import profit_pilot.utils.io as io_module

# Reload project modules so stale cached versions are replaced.
download_ohlcv_module = importlib.reload(download_ohlcv_module)
multi_crypto_env_module = importlib.reload(multi_crypto_env_module)
build_features_module = importlib.reload(build_features_module)
io_module = importlib.reload(io_module)

fetch_symbol_ohlcv = download_ohlcv_module.fetch_symbol_ohlcv
save_symbol_csv = download_ohlcv_module.save_symbol_csv
MultiCryptoTradingEnv = multi_crypto_env_module.MultiCryptoTradingEnv
BASE_FEATURE_COLUMNS = build_features_module.BASE_FEATURE_COLUMNS
add_technical_indicators = build_features_module.add_technical_indicators
align_frames = build_features_module.align_frames
build_arrays = build_features_module.build_arrays
bundle_directory = io_module.bundle_directory
save_json = io_module.save_json
symbol_slug = io_module.symbol_slug

if 'requested_trade_value' not in inspect.getsource(MultiCryptoTradingEnv._apply_buy_actions):
    raise RuntimeError(
        'Loaded an old MultiCryptoTradingEnv without notional multi-asset actions. '
        'Restart the runtime and make sure the latest src/profit_pilot/env/multi_crypto_env.py is present.'
    )
if 'cash_target_allocation' not in inspect.getsource(MultiCryptoTradingEnv._validate_settings):
    raise RuntimeError(
        'Loaded an old MultiCryptoTradingEnv without explicit cash allocation actions. '
        'Restart the runtime and rerun the setup/import cells.'
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_random_seed(SEED)

# Exchange fallback order - Binance is blocked in some regions (HTTP 451).
EXCHANGE_IDS = [
    exchange_id.strip()
    for exchange_id in os.environ.get('PROFIT_PILOT_EXCHANGES', 'binanceus,kucoin,okx,bybit,binance').split(',')
    if exchange_id.strip()
]
EXCHANGE_ID = EXCHANGE_IDS[0]
SYMBOLS = ['BTC/USDT', 'ETH/USDT']
TIMEFRAME = '5m'
DOWNLOAD_LIMIT = 1000

DATA_START_ISO = '2024-01-01T00:00:00Z'
TEST_START_ISO  = '2026-04-01T00:00:00Z'
DATA_END_ISO    = '2026-04-28T23:55:00Z'

INITIAL_CASH = 10_000.0
LOOKBACK = 96
TOTAL_TIMESTEPS = int(os.environ.get('PROFIT_PILOT_TOTAL_TIMESTEPS', '1500000'))
TRAIN_MODEL = True
USE_LSTM = True
EVALUATION_DETERMINISTIC = True
FORCE_REDOWNLOAD = False
RUN_FINAL_FULL_WINDOW_RETRAIN = False

# T4 x2 speed path: one model updates on cuda:0 while multiple rollout envs collect samples in parallel.
SINGLE_T4_BASELINE_N_ENVS = 1
DEFAULT_TRAIN_N_ENVS = min(4, max(2, (os.cpu_count() or 2) // 2))
TRAIN_N_ENVS = int(os.environ.get('PROFIT_PILOT_TRAIN_N_ENVS', str(DEFAULT_TRAIN_N_ENVS)))
TRAIN_N_ENVS = max(SINGLE_T4_BASELINE_N_ENVS, TRAIN_N_ENVS)
PPO_N_STEPS = int(os.environ.get('PROFIT_PILOT_N_STEPS', '4096'))
PPO_BATCH_SIZE = int(os.environ.get('PROFIT_PILOT_BATCH_SIZE', '1024'))
CHECKPOINT_EVERY_TIMESTEPS = int(os.environ.get('PROFIT_PILOT_CHECKPOINT_EVERY_TIMESTEPS', '50000'))
VEC_ENV_BENCHMARK_STEPS = int(os.environ.get('PROFIT_PILOT_VEC_ENV_BENCHMARK_STEPS', '256'))
MIN_PARALLEL_SPEEDUP = float(os.environ.get('PROFIT_PILOT_MIN_PARALLEL_SPEEDUP', '1.05'))
REQUIRE_PARALLEL_ROLLOUT_SPEEDUP = os.environ.get('PROFIT_PILOT_REQUIRE_PARALLEL_SPEEDUP', '1').lower() not in {'0', 'false', 'no'}

rollout_samples = PPO_N_STEPS * TRAIN_N_ENVS
if rollout_samples % PPO_BATCH_SIZE:
    print(
        f'Warning: PPO_N_STEPS * TRAIN_N_ENVS = {rollout_samples:,} is not divisible by '
        f'PPO_BATCH_SIZE = {PPO_BATCH_SIZE:,}. Consider adjusting the batch size.'
    )

# All writable paths live under WORKING_ROOT so Kaggle's read-only input is untouched.
RAW_ROOT             = WORKING_ROOT / 'data' / 'raw_kaggle_t4x2_5m_2024_2026'
PROCESSED_TRAIN_ROOT = WORKING_ROOT / 'data' / 'processed_kaggle_t4x2_5m_train_2024_2026_cash'
PROCESSED_TEST_ROOT  = WORKING_ROOT / 'data' / 'processed_kaggle_t4x2_5m_eval_2026_0401_0428_cash'
PROCESSED_FULL_ROOT  = WORKING_ROOT / 'data' / 'processed_kaggle_t4x2_5m_full_2024_2026_cash'
MODEL_DIR            = WORKING_ROOT / 'models' / 'kaggle_t4x2_5m_cash_allocation'
REPORT_DIR           = WORKING_ROOT / 'reports' / 'kaggle_t4x2_5m_cash_allocation'
PLOT_DIR             = REPORT_DIR / 'plots'
for path in [RAW_ROOT, PROCESSED_TRAIN_ROOT, PROCESSED_TEST_ROOT, PROCESSED_FULL_ROOT,
             MODEL_DIR, REPORT_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = f"profit_pilot_{'recurrent_' if USE_LSTM else ''}ppo_5m_t4x2_cash_multi_asset"
MODEL_PATH = MODEL_DIR / MODEL_NAME

print(f'Exchange fallback order : {EXCHANGE_IDS}')
print(f'Model name             : {MODEL_NAME}')
print(f'Total timesteps        : {TOTAL_TIMESTEPS:,}')
print(f'Training envs          : {TRAIN_N_ENVS} (baseline single T4 envs: {SINGLE_T4_BASELINE_N_ENVS})')
print(f'Rollout samples/update : {rollout_samples:,}')
print(f'PPO batch size         : {PPO_BATCH_SIZE:,}')


2026-05-08 06:00:20.885590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778220020.909622     183 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778220020.917341     183 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778220020.937578     183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778220020.937596     183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778220020.937599     183 computation_placer.cc:177] computation placer alr

Exchange fallback order : ['binanceus', 'kucoin', 'okx', 'bybit', 'binance']
Model name             : profit_pilot_recurrent_ppo_5m_t4x2_cash_multi_asset
Total timesteps        : 1,500,000
Training envs          : 2 (baseline single T4 envs: 1)
Rollout samples/update : 8,192
PPO batch size         : 1,024


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Download and Cache 5m Market Data


In [9]:
def create_market_data_exchange(exchange_id: str):
    exchange_class = getattr(ccxt, exchange_id)
    exchange = exchange_class({'enableRateLimit': True})
    exchange.timeout = 30_000
    return exchange


def read_cached_frame(path: Path) -> pd.DataFrame | None:
    if not path.exists() or FORCE_REDOWNLOAD:
        return None

    frame = pd.read_csv(path)
    if frame.empty or 'datetime' not in frame.columns:
        return None

    frame['datetime'] = pd.to_datetime(frame['datetime'], utc=True)
    start = pd.Timestamp(DATA_START_ISO)
    end = pd.Timestamp(DATA_END_ISO)
    if frame['datetime'].min() <= start and frame['datetime'].max() >= end - pd.Timedelta(minutes=5):
        return frame[(frame['datetime'] >= start) & (frame['datetime'] <= end)].copy()
    return None


def load_or_download_symbol(exchange, exchange_id: str, symbol: str) -> pd.DataFrame:
    target_path = RAW_ROOT / exchange_id / TIMEFRAME / f'{symbol_slug(symbol)}.csv'
    cached = read_cached_frame(target_path)
    if cached is not None:
        print(f'Using cached {symbol} from {exchange_id}: {len(cached):,} rows')
        return cached

    print(f'Downloading {symbol} {TIMEFRAME} from {exchange_id}: {DATA_START_ISO} to {DATA_END_ISO}')
    frame = fetch_symbol_ohlcv(
        exchange=exchange,
        symbol=symbol,
        timeframe=TIMEFRAME,
        since=DATA_START_ISO,
        until=DATA_END_ISO,
        limit=DOWNLOAD_LIMIT,
    )
    frame['datetime'] = pd.to_datetime(frame['datetime'], utc=True)
    saved_path = save_symbol_csv(frame, RAW_ROOT, exchange_id, TIMEFRAME, symbol)
    print(f'Saved {len(frame):,} rows to {saved_path}')
    return frame


def download_all_symbols_from_exchange(exchange_id: str) -> dict[str, pd.DataFrame]:
    exchange = create_market_data_exchange(exchange_id)
    try:
        exchange.load_markets()
        available_symbols = set(exchange.symbols or [])
        missing_symbols = [symbol for symbol in SYMBOLS if symbol not in available_symbols]
        if missing_symbols:
            raise ValueError(f'{exchange_id} does not list these symbols: {missing_symbols}')

        return {
            symbol: load_or_download_symbol(exchange, exchange_id, symbol)
            for symbol in SYMBOLS
        }
    finally:
        if hasattr(exchange, 'close'):
            exchange.close()


full_symbol_frames = None
download_errors = {}
for candidate_exchange_id in EXCHANGE_IDS:
    print(f'\nTrying exchange: {candidate_exchange_id}')
    try:
        candidate_frames = download_all_symbols_from_exchange(candidate_exchange_id)
        full_symbol_frames = candidate_frames
        EXCHANGE_ID = candidate_exchange_id
        print(f'Using exchange for this run: {EXCHANGE_ID}')
        break
    except Exception as exc:
        error_message = f'{type(exc).__name__}: {str(exc)}'
        download_errors[candidate_exchange_id] = error_message
        print(f'{candidate_exchange_id} failed: {error_message[:500]}')

if full_symbol_frames is None:
    display(pd.DataFrame(download_errors.items(), columns=['exchange', 'error']))
    raise RuntimeError(
        'No exchange in EXCHANGE_IDS could download all requested symbols. '
        'Try changing EXCHANGE_IDS in the config cell, for example to ["kucoin", "okx", "bybit"].'
    )

coverage_rows = []
for symbol, frame in full_symbol_frames.items():
    coverage_rows.append({
        'exchange': EXCHANGE_ID,
        'symbol': symbol,
        'rows': len(frame),
        'first': frame['datetime'].min(),
        'last': frame['datetime'].max(),
    })
display(pd.DataFrame(coverage_rows))



Trying exchange: binanceus
Using cached BTC/USDT from binanceus: 244,512 rows
Using cached ETH/USDT from binanceus: 244,512 rows
Using exchange for this run: binanceus


,exchange,symbol,rows,first,last
0,binanceus,BTC/USDT,244512,2024-01-01 00:00:00+00:00,2026-04-28 23:55:00+00:00
1,binanceus,ETH/USDT,244512,2024-01-01 00:00:00+00:00,2026-04-28 23:55:00+00:00


## Build Train and Test Feature Bundles


In [10]:
def split_symbol_frames(frames: dict[str, pd.DataFrame], start_iso: str, end_iso: str, inclusive_end: bool) -> dict[str, pd.DataFrame]:
    start = pd.Timestamp(start_iso)
    end = pd.Timestamp(end_iso)
    split_frames = {}
    for symbol, frame in frames.items():
        working = frame.copy()
        working['datetime'] = pd.to_datetime(working['datetime'], utc=True)
        if inclusive_end:
            mask = (working['datetime'] >= start) & (working['datetime'] <= end)
        else:
            mask = (working['datetime'] >= start) & (working['datetime'] < end)
        split = working.loc[mask].sort_values('datetime').reset_index(drop=True)
        if split.empty:
            raise ValueError(f'No rows for {symbol} between {start_iso} and {end_iso}')
        split_frames[symbol] = split
    return split_frames


def build_normalized_feature_bundle(
    symbol_frames: dict[str, pd.DataFrame],
    processed_root: Path,
    source_start_iso: str,
    source_end_iso: str,
    label: str,
    normalization_stats: dict[str, np.ndarray] | None = None,
) -> tuple[dict, dict[str, np.ndarray]]:
    enriched_frames = {}
    for symbol, frame in symbol_frames.items():
        enriched = add_technical_indicators(frame)
        if enriched.empty:
            raise ValueError(f'Feature frame became empty after indicator warm-up: {symbol}')
        enriched_frames[symbol] = enriched

    timestamps, aligned_frames = align_frames(enriched_frames)
    price_array, raw_tech_array, feature_export = build_arrays(SYMBOLS, aligned_frames)

    if normalization_stats is None:
        tech_mean = raw_tech_array.mean(axis=0)
        tech_std = raw_tech_array.std(axis=0)
        tech_std = np.where(tech_std < 1e-8, 1.0, tech_std)
        normalization_stats = {'mean': tech_mean, 'std': tech_std}

    tech_array = ((raw_tech_array - normalization_stats['mean']) / normalization_stats['std']).astype(np.float32)
    price_array = price_array.astype(np.float32)

    output_dir = bundle_directory(processed_root, TIMEFRAME, SYMBOLS)
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(output_dir / 'price_array.npy', price_array)
    np.save(output_dir / 'tech_array.npy', tech_array)
    np.save(output_dir / 'tech_mean.npy', normalization_stats['mean'])
    np.save(output_dir / 'tech_std.npy', normalization_stats['std'])
    pd.DataFrame({'datetime': timestamps.astype(str)}).to_csv(output_dir / 'timestamps.csv', index=False)
    feature_export.to_parquet(output_dir / 'feature_frame.parquet', index=False)

    metadata = {
        'label': label,
        'symbols': SYMBOLS,
        'feature_columns_per_asset': BASE_FEATURE_COLUMNS,
        'n_assets': len(SYMBOLS),
        'n_features_per_asset': len(BASE_FEATURE_COLUMNS),
        'price_array_shape': list(price_array.shape),
        'tech_array_shape': list(tech_array.shape),
        'timeframe': TIMEFRAME,
        'source_since': source_start_iso,
        'source_until': source_end_iso,
        'first_timestamp_after_warmup': str(timestamps.min()),
        'last_timestamp': str(timestamps.max()),
        'normalization': 'standard_score_using_train_stats',
    }
    save_json(metadata, output_dir / 'metadata.json')

    bundle = {
        'root': output_dir,
        'price_array': price_array,
        'tech_array': tech_array,
        'timestamps': pd.DataFrame({'datetime': timestamps.astype(str)}),
        'metadata': metadata,
    }
    return bundle, normalization_stats


train_frames = split_symbol_frames(full_symbol_frames, DATA_START_ISO, TEST_START_ISO, inclusive_end=False)
test_frames  = split_symbol_frames(full_symbol_frames, TEST_START_ISO,  DATA_END_ISO,  inclusive_end=True)

train_bundle, train_norm_stats = build_normalized_feature_bundle(
    train_frames,
    PROCESSED_TRAIN_ROOT,
    DATA_START_ISO,
    TEST_START_ISO,
    label='train_2024_0101_2026_0331',
)
test_bundle, _ = build_normalized_feature_bundle(
    test_frames,
    PROCESSED_TEST_ROOT,
    TEST_START_ISO,
    DATA_END_ISO,
    label='test_2026_0401_0428',
    normalization_stats=train_norm_stats,
)

bundle_summary = pd.DataFrame([train_bundle['metadata'], test_bundle['metadata']])
display(bundle_summary[[
    'label',
    'timeframe',
    'source_since',
    'source_until',
    'first_timestamp_after_warmup',
    'last_timestamp',
    'price_array_shape',
    'tech_array_shape',
]])


,label,timeframe,source_since,source_until,first_timestamp_after_warmup,last_timestamp,price_array_shape,tech_array_shape
0,train_2024_0101_2026_0331,5m,2024-01-01T00:00:00Z,2026-04-01T00:00:00Z,2024-01-01 08:00:00+00:00,2026-03-31 23:55:00+00:00,"[236351, 2]","[236351, 44]"
1,test_2026_0401_0428,5m,2026-04-01T00:00:00Z,2026-04-28T23:55:00Z,2026-04-01 08:00:00+00:00,2026-04-28 23:55:00+00:00,"[7968, 2]","[7968, 44]"


## Train Multi-Asset PPO on Kaggle T4 x2


In [11]:
ENV_KWARGS = dict(
    lookback=LOOKBACK,
    initial_cash=INITIAL_CASH,
    buy_cost_pct=0.001,
    sell_cost_pct=0.001,
    cash_norm=0.0001,
    holdings_norm=1.0,
    tech_norm=1.0,
    reward_scaling=1.0,
    reward_mode='net_log_return_alpha',
    action_mode='cash_target_allocation',
    cost_penalty_weight=0.10,
    risk_penalty_weight=0.10,
    alpha_reward_weight=1.0,
    drawdown_penalty_weight=0.20,
    risk_halt_penalty_weight=0.05,
    volatility_reward_weight=0.0,
    volatility_window=288,
    risk_adjusted_return_clip=5.0,
    max_position_fraction=0.60,
    max_gross_exposure=0.80,
    max_trade_fraction=0.50,
    stop_loss_pct=0.07,
    max_drawdown_pct=0.25,
    min_trade_quantity=0.0001,
    cooldown_steps=6,
    trade_deadband=0.10,
    rebalance_threshold=0.08,
    min_trade_notional=50.0,
    turnover_penalty_weight=0.02,
)

TRAIN_ENV_KWARGS = dict(ENV_KWARGS, random_start=True, episode_length=12 * 24 * 10)


def make_trading_env(bundle: dict, training: bool = False) -> MultiCryptoTradingEnv:
    if bundle['price_array'].shape[0] <= LOOKBACK + 2:
        raise ValueError('Not enough rows for the configured LOOKBACK.')
    env_kwargs = TRAIN_ENV_KWARGS if training else ENV_KWARGS
    return MultiCryptoTradingEnv(
        price_array=bundle['price_array'],
        tech_array=bundle['tech_array'],
        tickers=SYMBOLS,
        **env_kwargs,
    )


def _make_env_factory(bundle: dict, rank: int, training: bool = True):
    def _make_env():
        env = make_trading_env(bundle, training=training)
        env.reset(seed=SEED + rank)
        return env

    return _make_env


def make_vec_env(bundle: dict, n_envs: int = TRAIN_N_ENVS):
    env_fns = [_make_env_factory(bundle, rank=index, training=True) for index in range(n_envs)]
    if n_envs <= 1:
        return VecMonitor(DummyVecEnv(env_fns))
    if os.name == 'nt':
        print('SubprocVecEnv with fork is unavailable on Windows; using DummyVecEnv locally.')
        return VecMonitor(DummyVecEnv(env_fns))
    return VecMonitor(SubprocVecEnv(env_fns, start_method='fork'))


def benchmark_vec_env_throughput(bundle: dict, n_envs: int, steps: int = VEC_ENV_BENCHMARK_STEPS) -> dict:
    probe_env = make_vec_env(bundle, n_envs=n_envs)
    started_at = time.perf_counter()
    try:
        probe_env.reset()
        for _ in range(steps):
            actions = np.stack([probe_env.action_space.sample() for _ in range(probe_env.num_envs)])
            probe_env.step(actions)
        elapsed_sec = max(time.perf_counter() - started_at, 1e-9)
        env_steps = steps * probe_env.num_envs
        return {
            'n_envs': int(probe_env.num_envs),
            'steps_per_env': int(steps),
            'env_steps': int(env_steps),
            'elapsed_sec': float(elapsed_sec),
            'env_steps_per_sec': float(env_steps / elapsed_sec),
            'vec_env_class': 'SubprocVecEnv' if n_envs > 1 and os.name != 'nt' else 'DummyVecEnv',
        }
    finally:
        probe_env.close()


sanity_env = make_trading_env(train_bundle)
sanity_env.reset()
sanity_action = np.ones(sanity_env.action_space.shape, dtype=np.float32)
_, _, _, _, sanity_info = sanity_env.step(sanity_action)
sanity_positions = sanity_env.holdings * sanity_env.price_array[sanity_env.time]
sanity_exposures = sanity_positions / max(float(sanity_info['portfolio_value']), 1e-12)
sanity_table = pd.DataFrame({
    'symbol': SYMBOLS,
    'holding_after_positive_action': sanity_env.holdings,
    'position_value_after_positive_action': sanity_positions,
    'exposure_after_positive_action': sanity_exposures,
})
display(sanity_table)

if sanity_env.action_space.shape[0] != len(SYMBOLS) + 1:
    raise RuntimeError('Expected explicit cash allocation action plus one action per asset.')
if not np.all(sanity_positions > 1.0):
    raise RuntimeError(
        'Multi-asset sanity check failed: a positive asset action did not create notional exposure for every asset. '
        'Restart the runtime and rerun the setup/import cells.'
    )

benchmark_candidates = sorted({SINGLE_T4_BASELINE_N_ENVS, TRAIN_N_ENVS})
TRAIN_ENV_BENCHMARK_ROWS = [
    benchmark_vec_env_throughput(train_bundle, n_envs=candidate)
    for candidate in benchmark_candidates
]
benchmark_df = pd.DataFrame(TRAIN_ENV_BENCHMARK_ROWS)
display(benchmark_df)

baseline_rate = float(benchmark_df.loc[benchmark_df['n_envs'] == SINGLE_T4_BASELINE_N_ENVS, 'env_steps_per_sec'].iloc[0])
active_rate = float(benchmark_df.loc[benchmark_df['n_envs'] == TRAIN_N_ENVS, 'env_steps_per_sec'].iloc[0])
ROLLOUT_SPEEDUP_VS_SINGLE_ENV = active_rate / max(baseline_rate, 1e-9)
TRAIN_ENV_BENCHMARK = {
    'baseline_n_envs': SINGLE_T4_BASELINE_N_ENVS,
    'train_n_envs': TRAIN_N_ENVS,
    'rollout_speedup_vs_single_env': ROLLOUT_SPEEDUP_VS_SINGLE_ENV,
    'rows': TRAIN_ENV_BENCHMARK_ROWS,
}
print(f'Parallel rollout speedup vs single-env T4 baseline: {ROLLOUT_SPEEDUP_VS_SINGLE_ENV:.2f}x')

if TRAIN_N_ENVS > SINGLE_T4_BASELINE_N_ENVS and ROLLOUT_SPEEDUP_VS_SINGLE_ENV < MIN_PARALLEL_SPEEDUP:
    speed_message = (
        f'Parallel rollout benchmark was only {ROLLOUT_SPEEDUP_VS_SINGLE_ENV:.2f}x, below the required '
        f'{MIN_PARALLEL_SPEEDUP:.2f}x speedup. This session will not reliably finish earlier than the old '
        'single-env T4 notebook. Try a fresh Kaggle T4 x2 session, set PROFIT_PILOT_TRAIN_N_ENVS=2 or 4, '
        'or set PROFIT_PILOT_REQUIRE_PARALLEL_SPEEDUP=0 to continue anyway.'
    )
    if REQUIRE_PARALLEL_ROLLOUT_SPEEDUP:
        raise RuntimeError(speed_message)
    print(f'Warning: {speed_message}')

train_env = make_vec_env(train_bundle, n_envs=TRAIN_N_ENVS)
print(f'Observation shape: {train_env.observation_space.shape}')
print(f'Action shape     : {train_env.action_space.shape}')
print(f'Vectorized envs  : {train_env.num_envs}')


,symbol,holding_after_positive_action,position_value_after_positive_action,exposure_after_positive_action
0,BTC/USDT,0.078037,3333.333252,0.333556
1,ETH/USDT,1.441442,3333.333496,0.333556


,n_envs,steps_per_env,env_steps,elapsed_sec,env_steps_per_sec,vec_env_class
0,1,256,256,0.235629,1086.455955,DummyVecEnv
1,2,256,512,0.396211,1292.239961,SubprocVecEnv


Parallel rollout speedup vs single-env T4 baseline: 1.19x
Observation shape: (4229,)
Action shape     : (3,)
Vectorized envs  : 2


## Optional: True T4 x2 Feed-Forward Distributed PPO

Set `RUN_DISTRIBUTED_FEEDFORWARD_PPO = True` to train one shared feed-forward PPO policy across both Kaggle T4 GPUs with PyTorch DDP. This is separate from the SB3/RecurrentPPO path below.


In [ ]:
RUN_DISTRIBUTED_FEEDFORWARD_PPO = False
DISTRIBUTED_FEEDFORWARD_MODEL_NAME = 'profit_pilot_distributed_ppo_5m_t4x2_feedforward_cash_multi_asset'
DISTRIBUTED_ENVS_PER_RANK = int(os.environ.get('PROFIT_PILOT_DDP_ENVS_PER_RANK', '2'))
DISTRIBUTED_ROLLOUT_STEPS = int(os.environ.get('PROFIT_PILOT_DDP_ROLLOUT_STEPS', str(PPO_N_STEPS)))
DISTRIBUTED_TOTAL_TIMESTEPS = int(os.environ.get('PROFIT_PILOT_DDP_TOTAL_TIMESTEPS', str(TOTAL_TIMESTEPS)))
DISTRIBUTED_UPDATE_EPOCHS = int(os.environ.get('PROFIT_PILOT_DDP_UPDATE_EPOCHS', '10'))

if RUN_DISTRIBUTED_FEEDFORWARD_PPO:
    import subprocess
    import yaml

    if GPU_COUNT < 2:
        raise RuntimeError('Distributed feed-forward PPO needs Kaggle GPU T4 x2 with two visible CUDA devices.')

    distributed_config_path = WORKING_ROOT / 'config' / 'kaggle_t4x2_distributed_ppo.yaml'
    distributed_config_path.parent.mkdir(parents=True, exist_ok=True)
    with (PROJECT_ROOT / 'config' / 'project_config_5m_train.yaml').open('r', encoding='utf-8') as handle:
        distributed_config = yaml.safe_load(handle)

    distributed_config['data']['raw_dir'] = str(RAW_ROOT)
    distributed_config['data']['processed_dir'] = str(PROCESSED_TRAIN_ROOT)
    distributed_config['data']['until'] = TEST_START_ISO
    distributed_config['environment'].update(TRAIN_ENV_KWARGS)
    distributed_config['training']['use_lstm'] = False
    distributed_config['training']['device'] = 'auto'
    distributed_config['training']['model_dir'] = str(MODEL_DIR / 'distributed_feedforward')
    distributed_config['training']['report_dir'] = str(REPORT_DIR / 'distributed_feedforward')
    distributed_config['training']['total_timesteps'] = DISTRIBUTED_TOTAL_TIMESTEPS
    distributed_config['training']['n_steps'] = DISTRIBUTED_ROLLOUT_STEPS
    distributed_config['training']['batch_size'] = PPO_BATCH_SIZE

    with distributed_config_path.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(distributed_config, handle, sort_keys=False)

    distributed_command = [
        sys.executable,
        '-m',
        'torch.distributed.run',
        '--nproc_per_node=2',
        '-m',
        'profit_pilot.train.distributed_ppo',
        '--config',
        str(distributed_config_path),
        '--output-dir',
        str(MODEL_DIR / 'distributed_feedforward'),
        '--model-name',
        DISTRIBUTED_FEEDFORWARD_MODEL_NAME,
        '--envs-per-rank',
        str(DISTRIBUTED_ENVS_PER_RANK),
        '--rollout-steps',
        str(DISTRIBUTED_ROLLOUT_STEPS),
        '--total-timesteps',
        str(DISTRIBUTED_TOTAL_TIMESTEPS),
        '--batch-size',
        str(PPO_BATCH_SIZE),
        '--update-epochs',
        str(DISTRIBUTED_UPDATE_EPOCHS),
    ]
    print('Launching distributed feed-forward PPO:')
    print(' '.join(distributed_command))
    subprocess.run(distributed_command, check=True, cwd=str(PROJECT_ROOT))
    raise SystemExit(
        'Distributed feed-forward PPO finished. Stop here; the SB3/RecurrentPPO cells below are for the separate SB3 path.'
    )
else:
    print('Skipped distributed feed-forward PPO. Set RUN_DISTRIBUTED_FEEDFORWARD_PPO=True to use both T4 GPUs for one shared feed-forward model.')


In [12]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_EVERY_TIMESTEPS // max(TRAIN_N_ENVS, 1), 1),
    save_path=str(MODEL_DIR / 'checkpoints'),
    name_prefix=MODEL_NAME,
)

common_model_kwargs = dict(
    env=train_env,
    verbose=1,
    learning_rate=3e-5,
    n_steps=PPO_N_STEPS,
    batch_size=PPO_BATCH_SIZE,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.08,
    ent_coef=0.001,
    target_kl=0.02,
    device=DEVICE,
    tensorboard_log=str(REPORT_DIR / 'tensorboard'),
)

if TRAIN_MODEL:
    if USE_LSTM:
        model = RecurrentPPO('MlpLstmPolicy', **common_model_kwargs)
    else:
        model = PPO('MlpPolicy', policy_kwargs={'net_arch': [256, 256]}, **common_model_kwargs)

    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=checkpoint_callback, progress_bar=True)
    model.save(MODEL_PATH)
    train_env.close()

    training_summary = {
        'model_path': str(MODEL_PATH) + '.zip',
        'model_name': MODEL_NAME,
        'use_lstm': USE_LSTM,
        'device': DEVICE,
        'cuda_train_device': CUDA_TRAIN_DEVICE,
        'available_gpus': AVAILABLE_GPU_NAMES,
        'total_timesteps': TOTAL_TIMESTEPS,
        'symbols': SYMBOLS,
        'timeframe': TIMEFRAME,
        'train_window': [DATA_START_ISO, TEST_START_ISO],
        'test_window': [TEST_START_ISO, DATA_END_ISO],
        'lookback': LOOKBACK,
        'train_price_array_shape': list(train_bundle['price_array'].shape),
        'train_tech_array_shape': list(train_bundle['tech_array'].shape),
        'environment': ENV_KWARGS,
        'n_envs': TRAIN_N_ENVS,
        'single_t4_baseline_n_envs': SINGLE_T4_BASELINE_N_ENVS,
        'rollout_samples_per_update': PPO_N_STEPS * TRAIN_N_ENVS,
        'parallel_env_benchmark': TRAIN_ENV_BENCHMARK,
        'training_hyperparameters': {
            key: value
            for key, value in common_model_kwargs.items()
            if key not in {'env'}
        },
    }
    save_json(training_summary, MODEL_DIR / f'{MODEL_NAME}_training_summary.json')
else:
    model_class = RecurrentPPO if USE_LSTM else PPO
    model = model_class.load(MODEL_PATH, device=DEVICE)
    train_env.close()


Using cuda:0 device
Logging to /kaggle/working/reports/kaggle_t4x2_5m_cash_allocation/tensorboard/RecurrentPPO_2


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.1e+03  |
|    ep_rew_mean     | -49.6    |
| time/              |          |
|    fps             | 525      |
|    iterations      | 1        |
|    time_elapsed    | 15       |
|    total_timesteps | 8192     |
---------------------------------


KeyboardInterrupt: 

## Evaluate the Model Out of Sample


In [ ]:
def make_eval_record(env: MultiCryptoTradingEnv, timestamps: pd.Series, step: int, reward: float, action, info: dict) -> dict:
    action_dim = env.action_space.shape[0]
    action_array = np.zeros(action_dim, dtype=np.float32) if action is None else np.asarray(action, dtype=np.float32).reshape(-1)
    asset_action_offset = 1 if env.action_mode == 'cash_target_allocation' else 0
    target_weights = np.asarray(info.get('target_weights', np.zeros(len(SYMBOLS), dtype=np.float32)), dtype=np.float32).reshape(-1)
    executed_action_array = np.asarray(info.get('executed_actions', target_weights), dtype=np.float32).reshape(-1)
    record = {
        'step': step,
        'datetime': pd.to_datetime(timestamps.iloc[env.time], utc=True),
        'portfolio_value': float(env.portfolio_value),
        'benchmark_value': float(info.get('benchmark_value', env.equal_weight_value)),
        'cash': float(env.cash),
        'cash_weight': float(info.get('cash_weight', env.cash / max(env.portfolio_value, 1e-12))),
        'target_cash_weight': float(info.get('target_cash_weight', 0.0)),
        'reward': float(reward),
        'portfolio_return': float(info.get('portfolio_return', 0.0)),
        'drawdown': float(info.get('drawdown', 0.0)),
        'total_fees': float(info.get('total_fees', 0.0)),
        'turnover_penalty': float(info.get('turnover_penalty', 0.0)),
        'rolling_volatility': float(info.get('rolling_volatility', 0.0)),
        'risk_adjusted_return': float(info.get('risk_adjusted_return', 0.0)),
        'cost_penalty': float(info.get('cost_penalty', 0.0)),
        'risk_halted': bool(info.get('risk_halted', False)),
    }
    if asset_action_offset:
        record['action_cash'] = float(action_array[0]) if action_array.size else 0.0
    for index, symbol in enumerate(SYMBOLS):
        slug = symbol_slug(symbol)
        price = float(env.price_array[env.time, index])
        holding = float(env.holdings[index])
        action_index = index + asset_action_offset
        record[f'price_{slug}'] = price
        record[f'holding_{slug}'] = holding
        record[f'position_value_{slug}'] = holding * price
        record[f'action_{slug}'] = float(action_array[action_index]) if action_index < len(action_array) else 0.0
        record[f'target_weight_{slug}'] = float(target_weights[index]) if index < len(target_weights) else 0.0
        record[f'executed_action_{slug}'] = float(executed_action_array[index]) if index < len(executed_action_array) else 0.0
    return record


def evaluate_trading_model(model, bundle: dict, label: str) -> pd.DataFrame:
    env = make_trading_env(bundle)
    timestamps = pd.to_datetime(bundle['timestamps']['datetime'], utc=True)
    observation, info = env.reset()

    records = [make_eval_record(env, timestamps, step=0, reward=0.0, action=None, info=info)]
    done = False
    step = 0
    lstm_state = None
    episode_start = np.array([True], dtype=bool)

    while not done:
        if USE_LSTM:
            action, lstm_state = model.predict(
                observation,
                state=lstm_state,
                episode_start=episode_start,
                deterministic=EVALUATION_DETERMINISTIC,
            )
        else:
            action, _ = model.predict(observation, deterministic=EVALUATION_DETERMINISTIC)

        observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        step += 1
        records.append(make_eval_record(env, timestamps, step=step, reward=reward, action=action, info=info))
        episode_start = np.array([done], dtype=bool)

    result = pd.DataFrame(records)
    result['label'] = label
    return result


test_results = evaluate_trading_model(model, test_bundle, label='test_2026_0401_0428')
test_csv_path = REPORT_DIR / f'{MODEL_NAME}_test_results.csv'
test_results.to_csv(test_csv_path, index=False)
display(test_results.head())
display(test_results.tail())


## Metrics for Worthiness Check


In [ ]:
PERIODS_PER_YEAR = 365 * 24 * 12
RETURN_EPS = 1e-12


def add_curve_columns(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy()
    enriched['agent_return'] = enriched['portfolio_value'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    enriched['benchmark_return'] = enriched['benchmark_value'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    enriched['agent_cumulative_return'] = enriched['portfolio_value'] / enriched['portfolio_value'].iloc[0] - 1.0
    enriched['benchmark_cumulative_return'] = enriched['benchmark_value'] / enriched['benchmark_value'].iloc[0] - 1.0
    enriched['agent_drawdown'] = enriched['portfolio_value'] / enriched['portfolio_value'].cummax() - 1.0
    enriched['benchmark_drawdown'] = enriched['benchmark_value'] / enriched['benchmark_value'].cummax() - 1.0
    position_cols = [f'position_value_{symbol_slug(symbol)}' for symbol in SYMBOLS]
    enriched['gross_exposure'] = enriched[position_cols].sum(axis=1) / enriched['portfolio_value'].clip(lower=RETURN_EPS)
    for symbol in SYMBOLS:
        slug = symbol_slug(symbol)
        enriched[f'exposure_{slug}'] = enriched[f'position_value_{slug}'] / enriched['portfolio_value'].clip(lower=RETURN_EPS)
    return enriched


def annualized_return(values: pd.Series, timestamps: pd.Series) -> float:
    elapsed_seconds = max((timestamps.iloc[-1] - timestamps.iloc[0]).total_seconds(), 1.0)
    years = elapsed_seconds / (365 * 24 * 60 * 60)
    return float((values.iloc[-1] / max(values.iloc[0], RETURN_EPS)) ** (1 / years) - 1)


def sharpe_ratio(returns: pd.Series) -> float:
    std = returns.std(ddof=0)
    if std <= RETURN_EPS:
        return 0.0
    return float((returns.mean() / std) * math.sqrt(PERIODS_PER_YEAR))


def sortino_ratio(returns: pd.Series) -> float:
    downside = returns[returns < 0]
    downside_std = downside.std(ddof=0)
    if downside_std <= RETURN_EPS or downside.empty:
        return 0.0
    return float((returns.mean() / downside_std) * math.sqrt(PERIODS_PER_YEAR))


def summarize_strategy(frame: pd.DataFrame, value_col: str, return_col: str, drawdown_col: str) -> dict:
    returns = frame[return_col]
    values = frame[value_col]
    max_drawdown = abs(float(frame[drawdown_col].min()))
    ann_return = annualized_return(values, frame['datetime'])
    ann_vol = float(returns.std(ddof=0) * math.sqrt(PERIODS_PER_YEAR))
    return {
        'initial_value': float(values.iloc[0]),
        'final_value': float(values.iloc[-1]),
        'total_return_pct': float((values.iloc[-1] / values.iloc[0] - 1.0) * 100),
        'annualized_return_pct': ann_return * 100,
        'annualized_volatility_pct': ann_vol * 100,
        'sharpe': sharpe_ratio(returns),
        'sortino': sortino_ratio(returns),
        'max_drawdown_pct': max_drawdown * 100,
        'calmar': float(ann_return / max(max_drawdown, RETURN_EPS)),
        'win_rate_pct': float((returns > 0).mean() * 100),
    }


test_results = add_curve_columns(test_results)
test_results.to_csv(test_csv_path, index=False)

holding_cols = [f'holding_{symbol_slug(symbol)}' for symbol in SYMBOLS]
trade_events = int((test_results[holding_cols].diff().abs().sum(axis=1) > 1e-8).sum())
active_assets = int(sum((test_results[f'holding_{symbol_slug(symbol)}'].abs() > 1e-12).any() for symbol in SYMBOLS))

agent_metrics = summarize_strategy(test_results, 'portfolio_value', 'agent_return', 'agent_drawdown')
benchmark_metrics = summarize_strategy(test_results, 'benchmark_value', 'benchmark_return', 'benchmark_drawdown')
agent_metrics.update({
    'total_fees': float(test_results['total_fees'].sum()),
    'trade_events': trade_events,
    'active_assets_traded': active_assets,
    'mean_gross_exposure_pct': float(test_results['gross_exposure'].mean() * 100),
    'risk_halted': bool(test_results['risk_halted'].any()),
})
benchmark_metrics.update({
    'total_fees': 0.0,
    'trade_events': 0,
    'mean_gross_exposure_pct': 100.0,
    'risk_halted': False,
})

metrics_df = pd.DataFrame({'agent': agent_metrics, 'equal_weight_benchmark': benchmark_metrics}).T
metrics_df['alpha_vs_benchmark_pct'] = metrics_df['total_return_pct'] - metrics_df.loc['equal_weight_benchmark', 'total_return_pct']
display(metrics_df.round(4))

metrics_path = REPORT_DIR / f'{MODEL_NAME}_test_metrics.json'
metrics_payload = json.loads(metrics_df.reset_index().rename(columns={'index': 'strategy'}).to_json(orient='records'))
with metrics_path.open('w', encoding='utf-8') as handle:
    json.dump(metrics_payload, handle, indent=2)

agent = metrics_df.loc['agent']
benchmark = metrics_df.loc['equal_weight_benchmark']
passes_return   = agent['total_return_pct'] > benchmark['total_return_pct']
passes_sharpe   = agent['sharpe'] > benchmark['sharpe']
passes_drawdown = agent['max_drawdown_pct'] <= benchmark['max_drawdown_pct']
passes_positive = agent['total_return_pct'] > 0
score   = sum([passes_return, passes_sharpe, passes_drawdown, passes_positive])
verdict = 'PASS for deeper research' if score >= 3 else 'NOT WORTH DEPLOYING YET'
print(f'Worthiness check: {verdict} ({score}/4 conditions passed)')
print(f'Saved enriched test results: {test_csv_path}')


## Plots for Model Analysis


In [ ]:
sns.set_theme(style='whitegrid')

plot_frame = test_results.copy()
plot_frame['datetime'] = pd.to_datetime(plot_frame['datetime'], utc=True)
rolling_window = 12 * 24 * 7
plot_frame['agent_7d_return'] = plot_frame['portfolio_value'] / plot_frame['portfolio_value'].shift(rolling_window) - 1.0
plot_frame['benchmark_7d_return'] = plot_frame['benchmark_value'] / plot_frame['benchmark_value'].shift(rolling_window) - 1.0

fig, axes = plt.subplots(5, 1, figsize=(15, 20), sharex=True)

axes[0].plot(plot_frame['datetime'], plot_frame['portfolio_value'], label='Agent Portfolio', linewidth=1.3)
axes[0].plot(plot_frame['datetime'], plot_frame['benchmark_value'], label='Equal-Weight Benchmark', linewidth=1.1)
axes[0].set_title('Portfolio Value')
axes[0].set_ylabel('USDT')
axes[0].legend()

axes[1].plot(plot_frame['datetime'], plot_frame['agent_cumulative_return'] * 100, label='Agent', linewidth=1.2)
axes[1].plot(plot_frame['datetime'], plot_frame['benchmark_cumulative_return'] * 100, label='Benchmark', linewidth=1.2)
axes[1].set_title('Cumulative Return')
axes[1].set_ylabel('%')
axes[1].legend()

axes[2].fill_between(plot_frame['datetime'], plot_frame['agent_drawdown'] * 100, 0, alpha=0.35, label='Agent')
axes[2].plot(plot_frame['datetime'], plot_frame['benchmark_drawdown'] * 100, label='Benchmark', linewidth=1.0)
axes[2].set_title('Drawdown')
axes[2].set_ylabel('%')
axes[2].legend()

for symbol in SYMBOLS:
    slug = symbol_slug(symbol)
    axes[3].plot(plot_frame['datetime'], plot_frame[f'exposure_{slug}'] * 100, label=f'{symbol} exposure', linewidth=1.0)
axes[3].plot(plot_frame['datetime'], plot_frame['gross_exposure'] * 100, label='Gross exposure', linewidth=1.2, linestyle='--')
axes[3].set_title('Portfolio Exposure')
axes[3].set_ylabel('% of portfolio')
axes[3].legend()

for symbol in SYMBOLS:
    slug = symbol_slug(symbol)
    axes[4].plot(plot_frame['datetime'], plot_frame[f'action_{slug}'], label=f'{symbol} action', linewidth=0.8, alpha=0.8)
axes[4].axhline(0, color='black', linewidth=0.8)
axes[4].set_title('Model Actions')
axes[4].set_ylabel('Action')
axes[4].legend()

plt.tight_layout()
combined_plot_path = PLOT_DIR / f'{MODEL_NAME}_test_analysis.png'
fig.savefig(combined_plot_path, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(plot_frame['datetime'], plot_frame['agent_7d_return'] * 100, label='Agent 7d return')
axes[0].plot(plot_frame['datetime'], plot_frame['benchmark_7d_return'] * 100, label='Benchmark 7d return')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Rolling 7-Day Return')
axes[0].set_ylabel('%')
axes[0].legend()

sns.histplot(plot_frame['agent_return'] * 100, bins=80, kde=True, ax=axes[1], label='Agent', color='tab:blue', stat='density')
sns.histplot(plot_frame['benchmark_return'] * 100, bins=80, kde=True, ax=axes[1], label='Benchmark', color='tab:orange', stat='density', alpha=0.45)
axes[1].set_title('5m Return Distribution')
axes[1].set_xlabel('Return %')
axes[1].legend()

plt.tight_layout()
risk_plot_path = PLOT_DIR / f'{MODEL_NAME}_risk_diagnostics.png'
fig.savefig(risk_plot_path, dpi=180, bbox_inches='tight')
plt.show()


## Artifact Summary


In [ ]:
artifact_summary = {
    'model_zip': str(MODEL_PATH) + '.zip',
    'training_summary': str(MODEL_DIR / f'{MODEL_NAME}_training_summary.json'),
    'test_results_csv': str(test_csv_path),
    'test_metrics_json': str(metrics_path),
    'combined_analysis_plot': str(combined_plot_path),
    'risk_diagnostics_plot': str(risk_plot_path),
    'train_processed_bundle': str(train_bundle['root']),
    'test_processed_bundle': str(test_bundle['root']),
}
display(pd.DataFrame(artifact_summary.items(), columns=['artifact', 'path']))
with (REPORT_DIR / f'{MODEL_NAME}_artifact_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_summary, handle, indent=2)
print(f"Saved artifact summary: {REPORT_DIR / f'{MODEL_NAME}_artifact_summary.json'}")
print(f'\nAll outputs saved under: {WORKING_ROOT}')
print('Use Kaggle Output tab or Save Version to persist these files.')


## Optional: Final Full-Window Retrain

Set `RUN_FINAL_FULL_WINDOW_RETRAIN = True` in the config cell (cell 4) once the worthiness check passes.


In [ ]:
if RUN_FINAL_FULL_WINDOW_RETRAIN:
    full_frames = split_symbol_frames(full_symbol_frames, DATA_START_ISO, DATA_END_ISO, inclusive_end=True)
    full_bundle, _ = build_normalized_feature_bundle(
        full_frames,
        PROCESSED_FULL_ROOT,
        DATA_START_ISO,
        DATA_END_ISO,
        label='full_2024_0101_2026_0428',
    )
    full_env = make_vec_env(full_bundle, n_envs=TRAIN_N_ENVS)
    final_name = f'{MODEL_NAME}_full_window'
    final_path = MODEL_DIR / final_name
    final_model_kwargs = dict(
        env=full_env,
        verbose=1,
        learning_rate=3e-5,
        n_steps=PPO_N_STEPS,
        batch_size=PPO_BATCH_SIZE,
        gamma=0.995,
        gae_lambda=0.95,
        clip_range=0.08,
        ent_coef=0.001,
        target_kl=0.02,
        device=DEVICE,
        tensorboard_log=str(REPORT_DIR / 'tensorboard'),
    )
    if USE_LSTM:
        final_model = RecurrentPPO('MlpLstmPolicy', **final_model_kwargs)
    else:
        final_model = PPO(
            'MlpPolicy',
            policy_kwargs={'net_arch': [256, 256]},
            **final_model_kwargs,
        )
    final_model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
    final_model.save(final_path)
    full_env.close()
    print(f'Saved final full-window model: {final_path}.zip')
else:
    print('Skipped final full-window retrain. Set RUN_FINAL_FULL_WINDOW_RETRAIN = True after the test metrics are acceptable.')
